<a id="import"></a>
# <center>Import Need Modules</center>

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import seaborn as sns
sns.set_style('darkgrid')
import shutil
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Dense, Activation,Dropout,Conv2D, MaxPooling2D,LeakyReLU, BatchNormalization
from tensorflow.keras.optimizers import Adam, Adamax
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.metrics import categorical_crossentropy
from tensorflow.keras import regularizers
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
import time
from pathlib import Path
from tqdm import tqdm
import sys
if not sys.warnoptions:
    import warnings
    warnings.simplefilter("ignore")
pd.set_option('display.max_columns', None)  # or 1000
pd.set_option('display.max_rows', None)  # or 1000
pd.set_option('display.max_colwidth', None)  # or 199
print('All modules have been imported')

All modules have been imported


<a id="pc"></a>
## <center>Define a function to print text in specified rgb foreground and background colors</center>
### Add some PZAZZ to your printed output with this function  
form of the call is:  print_in_color(txt_msg, fore_tupple, back_tupple where:
* txt_msg is the string to be printed out  
* fore_tuple is tuple of the form (r,g,b) specifying the foreground color of the text
* back_tuple is tuple of the form (r,g,b) specifying the background color of the text

In [2]:
def print_in_color(txt_msg,fore_tupple=(0,255,255),back_tupple=(100,100,100)):
    #prints the text_msg in the foreground color specified by fore_tupple with the background specified by back_tupple
    #text_msg is the text, fore_tupple is foregroud color tupple (r,g,b), back_tupple is background tupple (r,g,b)
    # default parameter print in cyan foreground and gray background
    rf,gf,bf=fore_tupple
    rb,gb,bb=back_tupple
    msg='{0}' + txt_msg
    mat='\33[38;2;' + str(rf) +';' + str(gf) + ';' + str(bf) + ';48;2;' + str(rb) + ';' +str(gb) + ';' + str(bb) +'m'
    print(msg .format(mat), flush=True)
    print('\33[0m', flush=True) # returns default print color to back to black
    return

# example default print
msg='test of default colors'
print_in_color(msg)

test of default colors



In [3]:
dataset_dir = "archive/LEGO brick images v1"

# Define Data Augmentation


In [4]:
def create_augmentation_model():
    augment_model = keras.Sequential([
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.2),
        keras.layers.RandomZoom(0.2),
        keras.layers.RandomTranslation(height_factor=0.2, width_factor=0.2),
        keras.layers.RandomContrast(0.2),
        keras.layers.RandomBrightness(0.2)
    ])
    return augment_model

In [5]:
def make_data_sets(batch_size, train_dir, img_size, augmentation_layer):
    def augment(image, label):
        image = augmentation_layer(image, training=True)
        return image, label

    full_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        seed=123,
        image_size=img_size,
        batch_size=None,
        label_mode='categorical',
        shuffle=True
    )

    ds_class_names = full_ds.class_names

    dataset_size = full_ds.cardinality().numpy()
    if dataset_size < 0:
        dataset_size = sum(1 for _ in full_ds)

    full_ds = full_ds.shuffle(buffer_size=dataset_size, seed=123, reshuffle_each_iteration=False)

    train_size = int(0.7 * dataset_size)
    val_size   = int(0.15 * dataset_size)

    train_ds = full_ds.take(train_size)
    remaining = full_ds.skip(train_size)
    valid_ds  = remaining.take(val_size)
    test_ds   = remaining.skip(val_size)

    AUTOTUNE = tf.data.AUTOTUNE

    train_ds = (train_ds
                .map(augment, num_parallel_calls=AUTOTUNE)
                .shuffle(1000)
                .batch(batch_size)
                .prefetch(AUTOTUNE))

    valid_ds = (valid_ds
                .batch(batch_size)
                .cache()
                .prefetch(AUTOTUNE))

    test_ds  = (test_ds
                .batch(batch_size)
                .cache()
                .prefetch(AUTOTUNE))

    return train_ds, valid_ds, test_ds, ds_class_names

In [6]:
img_size=(224, 224)
batch_size=32

augmentation_model = create_augmentation_model()

train_ds, valid_ds, test_ds, class_names = make_data_sets(batch_size, dataset_dir, img_size, augmentation_model)

Found 6379 files belonging to 16 classes.


# Transfer Learning Model

In [7]:
def make_model(img_size, lr, class_count):
    img_shape = (img_size[0], img_size[1], 3)
    inputs = tf.keras.Input(shape=img_shape)

    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights='imagenet',
        pooling='max',
        input_tensor=inputs
    )

    base_model.trainable = True
    x = base_model.output

    x = BatchNormalization(axis=-1, momentum=0.99, epsilon=0.001)(x)
    x = Dense(256, kernel_regularizer=regularizers.l2(0.016))(x)
    x = LeakyReLU(alpha=0.1)(x)
    x = Dropout(rate=0.4)(x)
    outputs = Dense(class_count, activation='softmax')(x)

    model = Model(inputs, outputs)

    model.compile(optimizer=Adamax(learning_rate=lr),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

lr=1e-5
model=make_model(img_size=img_size, lr=lr, class_count=16)


# Instantiate custom callback

In [8]:
checkpoint = ModelCheckpoint(
    'best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

lr_reducer = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

callbacks = [checkpoint, lr_reducer, early_stop]

# Train model

In [9]:
history = model.fit(train_ds, validation_data=valid_ds, batch_size = batch_size, epochs=20, callbacks=callbacks)

Epoch 1/15
 37/140 ━━━━━━━━━━━━━━━━━━━━ 6:30 4s/step - accuracy: 0.0728 - loss: 10.6308

KeyboardInterrupt: 

<a id="plot"></a>
# <center>Define a function to plot the training data

In [ ]:
def tr_plot(tr_data):
    start_epoch=0
    #Plot the training and validation data
    tacc=tr_data.history['accuracy']
    tloss=tr_data.history['loss']
    vacc=tr_data.history['val_accuracy']
    vloss=tr_data.history['val_loss']
    Epoch_count=len(tacc)+ start_epoch
    Epochs=[]
    for i in range (start_epoch ,Epoch_count):
        Epochs.append(i+1)
    index_loss=np.argmin(vloss)#  this is the epoch with the lowest validation loss
    val_lowest=vloss[index_loss]
    index_acc=np.argmax(vacc)
    acc_highest=vacc[index_acc]
    plt.style.use('fivethirtyeight')
    sc_label='best epoch= '+ str(index_loss+1 +start_epoch)
    vc_label='best epoch= '+ str(index_acc + 1+ start_epoch)
    fig,axes=plt.subplots(nrows=1, ncols=2, figsize=(25,10))
    axes[0].plot(Epochs,tloss, 'r', label='Training loss')
    axes[0].plot(Epochs,vloss,'g',label='Validation loss' )
    axes[0].scatter(index_loss+1 +start_epoch,val_lowest, s=150, c= 'blue', label=sc_label)
    axes[0].scatter(Epochs, tloss, s=100, c='red')
    axes[0].set_title('Training and Validation Loss')
    axes[0].set_xlabel('Epochs', fontsize=18)
    axes[0].set_ylabel('Loss', fontsize=18)
    axes[0].legend()
    axes[1].plot (Epochs,tacc,'r',label= 'Training Accuracy')
    axes[1].scatter(Epochs, tacc, s=100, c='red')
    axes[1].plot (Epochs,vacc,'g',label= 'Validation Accuracy')
    axes[1].scatter(index_acc+1 +start_epoch,acc_highest, s=150, c= 'blue', label=vc_label)
    axes[1].set_title('Training and Validation Accuracy')
    axes[1].set_xlabel('Epochs', fontsize=18)
    axes[1].set_ylabel('Accuracy', fontsize=18)
    axes[1].legend()

    plt.tight_layout
    plt.show()
    return

tr_plot(history)

# Confusion Matrix and Classification Report


In [ ]:
true_labels_one_hot = np.concatenate([y for x, y in test_ds], axis=0)
true_labels = np.argmax(true_labels_one_hot, axis=1)

predictions = model.predict(test_ds)

pred_labels = np.argmax(predictions, axis=1)

cm = confusion_matrix(true_labels, pred_labels)

In [ ]:
def plot_confusion_matrix(cm, class_names, normalize=False):
    if normalize:
        cm_plot = cm.astype("float") / cm.sum(axis=1, keepdims=True)
        fmt = ".0%"
        title = "Normalized Confusion Matrix"
        vmin, vmax = 0, 1
    else:
        cm_plot = cm
        fmt = "d"
        title = "Confusion Matrix"
        vmin, vmax = None, None

    n = len(class_names)
    fig, ax = plt.subplots(figsize=(16, 14))

    sns.heatmap(
        cm_plot,
        annot=True,
        fmt=fmt,
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax,
        vmin=vmin,
        vmax=vmax,
        linewidths=0.5,
        linecolor="lightgrey",
        annot_kws={"size": 8},          # smaller annotation font
        cbar_kws={"shrink": 0.8}
    )

    # Highlight the diagonal (correct predictions) with a box
    for i in range(n):
        ax.add_patch(plt.Rectangle(
            (i, i), 1, 1,
            fill=False, edgecolor="red", lw=1.5
        ))

    ax.set_xlabel("Predicted Label", fontsize=13, labelpad=12)
    ax.set_ylabel("True Label", fontsize=13, labelpad=12)
    ax.set_title(title, fontsize=15, pad=15)

    # Rotate x labels and shrink font so they don't overlap
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()


plot_confusion_matrix(cm, class_names, normalize=True)

In [ ]:
print(classification_report(true_labels, pred_labels, target_names=class_names))

In [ ]:
# Test predictions

In [ ]:
from PIL import Image

image_dir = Path("test_set")
img_height, img_width = img_size

for filename in os.listdir(image_dir):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        file_path = os.path.join(image_dir, filename)

        try:
            img = Image.open(file_path).convert('RGB')
            img = img.resize((img_width, img_height))

            img_array = np.array(img) / 255.0
            img_array = np.expand_dims(img_array, axis=0)

            predictions = model.predict(img_array)

            print(f"Image: {filename} | Prediction: {predictions}")

        except Exception as e:
            print(f"Error processing {filename}: {e}")

<a id="save"></a>
# <center>Save the model </a>

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("model.tflite", 'wb') as f:
  f.write(tflite_model)